# UYKU_HIKAYESI_V3_FINAL_PRODUCTION — Google Colab Pro (T4 GPU)
Yalnızca 2 hücre: **Hücre 1** kurulum/doğrulama, **Hücre 2** uçtan uca üretim veya kaldığı yerden devam.


In [ ]:
# ==============================================================================
# UYKU_HIKAYESI_V3_FINAL_PRODUCTION — COLAB PRO T4 HÜCRE 1: KURULUM VE HAZIRLIK
# ==============================================================================
import hashlib, importlib.util, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

V3_DRIVE_ROOT = Path("/content/drive/MyDrive/UYKU_HIKAYESI_V3_FINAL_PRODUCTION")
REQ_FILE = V3_DRIVE_ROOT / "requirements_v3_colab_py313.txt"
RUNNER = V3_DRIVE_ROOT / "v3_runner.py"

assert V3_DRIVE_ROOT.exists(), f"V3 proje klasörü Drive'da bulunamadı: {V3_DRIVE_ROOT}"
assert RUNNER.exists(), f"Ana çalıştırıcı bulunamadı: {RUNNER}"

# 1. Yalnızca eksik paketleri kur (Colab CUDA PyTorch sürümünü EZMEZ)
check_mods = [
    ("whisper", "openai-whisper"),
    ("TTS", "coqui-tts"),
    ("pyloudnorm", "pyloudnorm"),
    ("pydub", "pydub"),
    ("soundfile", "soundfile"),
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("psutil", "psutil"),
]
missing = [pkg for mod, pkg in check_mods if importlib.util.find_spec(mod) is None]
if missing:
    print("Eksik paketler kuruluyor:", missing, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *missing])
else:
    print("Tüm zorunlu paketler zaten kurulu.", flush=True)

# 2. GPU ve ortam kontrolü
import torch
has_cuda = bool(getattr(torch.cuda, "is_available")())
assert has_cuda, "HATA: GPU aktif değil! Çalışma Zamanı -> T4 GPU seçin."
print("GPU:", torch.cuda.get_device_name(0), "| Python:", sys.version.split()[0], flush=True)

# 3. Kuru çalıştırma (dry-run) ile senaryo, manifest ve koruma doğrulaması
rc = subprocess.run([sys.executable, str(RUNNER), "--dry-run"], check=False).returncode
assert rc == 0, f"Dry-run doğrulaması başarısız (kod={rc})"
print("HAZIR: V3 ortamı, bağımlılıklar ve korunan dosyalar doğrulandı.", flush=True)


In [ ]:
# ==============================================================================
# UYKU_HIKAYESI_V3_FINAL_PRODUCTION — COLAB PRO T4 HÜCRE 2: ÜRETİMİ BAŞLAT / DEVAM ET
# ==============================================================================
import subprocess, sys
from pathlib import Path

V3_DRIVE_ROOT = Path("/content/drive/MyDrive/UYKU_HIKAYESI_V3_FINAL_PRODUCTION")
RUNNER = V3_DRIVE_ROOT / "v3_runner.py"

# İsteğe bağlı: başka bir senaryo için yolu değiştirin; None bırakılırsa varsayılan senaryo kullanılır.
SENARYO_YOLU = None
MAX_PARAGRAF = 0   # 0 = tüm senaryo (125+ paragraf); ilk doğrulama için 10 yapılabilir
STOP_AFTER = 0     # 0 = kesintisiz; kesinti/devam testi için 3 yapılabilir

cmd = [sys.executable, str(RUNNER), "--stages", "all"]
if SENARYO_YOLU:
    cmd += ["--script", SENARYO_YOLU]
if MAX_PARAGRAF > 0:
    cmd += ["--max-paragraphs", str(MAX_PARAGRAF)]
if STOP_AFTER > 0:
    cmd += ["--stop-after", str(STOP_AFTER)]

print("Çalıştırılıyor:", " ".join(cmd), flush=True)
res = subprocess.run(cmd, check=False)
print("V3 ÇIKIŞ KODU:", res.returncode, flush=True)
